In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

In [ ]:
# Load dataset
file_path = "company_esg_financial_dataset.csv"
df = pd.read_csv("/content/company_esg_financial_dataset.csv")
df

In [ ]:
# Create target: ESG Risk Label from ESG_Overall
conditions = [
    df['ESG_Overall'] < 40,
    (df['ESG_Overall'] >= 40) & (df['ESG_Overall'] < 70),
    df['ESG_Overall'] >= 70
]
choices = ['High Risk', 'Medium Risk', 'Low Risk']
df['ESG_Risk_Label'] = np.select(conditions, choices, default='Medium Risk')

In [ ]:
# Feature engineering
for col in ['Revenue', 'MarketCap', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption']:
    df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

df['Carbon_Intensity'] = df['CarbonEmissions'] / (df['Revenue'].replace(0, np.nan))
df['Energy_Intensity'] = df['EnergyConsumption'] / (df['Revenue'].replace(0, np.nan))
df['Water_Intensity'] = df['WaterUsage'] / (df['Revenue'].replace(0, np.nan))
df['ESG_Spread'] = df[['ESG_Environmental','ESG_Social','ESG_Governance']].max(axis=1) - df[['ESG_Environmental','ESG_Social','ESG_Governance']].min(axis=1)

feature_cols = [
    'Industry','Region','Year','Revenue','ProfitMargin','MarketCap','GrowthRate',
    'ESG_Environmental','ESG_Social','ESG_Governance',
    'CarbonEmissions','WaterUsage','EnergyConsumption',
    'log_Revenue','log_MarketCap','log_CarbonEmissions','log_WaterUsage','log_EnergyConsumption',
    'Carbon_Intensity','Energy_Intensity','Water_Intensity','ESG_Spread'
]

X = df[feature_cols]
y = df['ESG_Risk_Label']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=8,
    random_state=42,
    class_weight='balanced'
)

clf = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)
classes = clf.named_steps['model'].classes_

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))



In [ ]:
# Optional multiclass ROC-AUC
y_test_bin = label_binarize(y_test, classes=classes)
auc = roc_auc_score(y_test_bin, y_prob, average='weighted', multi_class='ovr')
print("ROC-AUC OVR:", auc)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
#Feature Importance
feature_names = clf.named_steps['preprocessor'].get_feature_names_out()
importances = clf.named_steps['model'].feature_importances_
fi = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(fi.head(15)['Feature'][::-1], fi.head(15)['Importance'][::-1], color='teal')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Risk Class Distribution
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='ESG_Risk_Label', order=['High Risk','Medium Risk','Low Risk'], palette='viridis')
plt.title('Distribution of ESG Risk Classes')
plt.tight_layout()
plt.show()

In [ ]:
# Profit Margin vs Risk
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='ESG_Risk_Label', y='ProfitMargin',
            order=['High Risk','Medium Risk','Low Risk'], palette='magma')
plt.title('Profit Margin by ESG Risk Class')
plt.tight_layout()
plt.show()